# GRIB: Interpolating from pressure to height levels

This notebook demonstrates how to interpolate GRIB fieldlist data on pressure levels to height levels. Fieldlist support requires the usage of earthkit-data.

In [ ]:
import earthkit.data as ekd

import earthkit.meteo.vertical.fieldlist as vertical

## Getting the data

The input is GRIB data on pressure levels. We fetch the file and inspect its contents.

In [ ]:
fl = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/earthkit-meteo/test-data/tz_pl.grib1",
).to_fieldlist()
fl.ls()

The file contains:

- **T** – temperature (K) on 9 pressure levels (1000, 850, 700, 500, 400, 300, 200, 150, 100 hPa)
- **Z** – geopotential (m²/s²) on the same 9 pressure levels and on the surface

We extract each quantity below.

In [ ]:
t = fl.sel({"parameter.variable": "t", "vertical.level_type": "pressure"})
z = fl.sel({"parameter.variable": "z", "vertical.level_type": "pressure"})
zs = fl.sel({"parameter.variable": "z", "vertical.level_type": "surface"})[0]  # surface geopotential

t.ls()

## Using interpolate_pressure_to_height_levels

### Geometric height above the ground

In [ ]:
target_h = [1000.0, 5000.0]  # geometric height (m), above the ground

t_res = vertical.interpolate_pressure_to_height_levels(
    t,  # data to interpolate
    target_h,
    z,  # geopotential on the same pressure levels
    zs=zs,  # surface geopotential
    h_type="geometric",
    h_reference="ground",
    interpolation="linear",
)
t_res.ls()

### Geometric height above sea level

In [ ]:
target_h_sea = [1000.0, 5000.0]  # geometric height (m), above sea level

# zs is not used when h_reference="sea"
t_res_sea = vertical.interpolate_pressure_to_height_levels(
    t,
    target_h_sea,
    z,
    h_type="geometric",
    h_reference="sea",
    interpolation="linear",
)
t_res_sea.ls()

### Geopotential height above the ground

In [ ]:
target_h_gp = [1000.0, 5000.0]  # geopotential height (m), above the ground

t_res_gp = vertical.interpolate_pressure_to_height_levels(
    t,
    target_h_gp,
    z,
    zs=zs,
    h_type="geopotential",
    h_reference="ground",
    interpolation="linear",
)
t_res_gp.ls()

### Using aux levels

The lowest pressure level in the input data is 1000 hPa. In grid points where the surface pressure is less than 100000 Pa (1000 hPa), this level lies below the surface and the corresponding geopotential height is undefined. Interpolating to a height just above the surface (e.g. 2 m) will therefore return NaN for those points:

In [ ]:
target_h_near = [2.0]  # m above the ground

t_res_nan = vertical.interpolate_pressure_to_height_levels(
    t,
    target_h_near,
    z,
    zs=zs,
    h_type="geometric",
    h_reference="ground",
    interpolation="linear",
)
t_res_nan.to_numpy()[:, :2, :2]

To overcome this problem we can supply auxiliary data at the surface via the `aux_bottom_*` kwargs. The example below prescribes a constant temperature at height 0 m (the surface).

In [ ]:
t_res_aux = vertical.interpolate_pressure_to_height_levels(
    t,
    target_h_near,
    z,
    zs=zs,
    h_type="geometric",
    h_reference="ground",
    aux_bottom_h=0.0,  # m — height of the auxiliary level (surface)
    aux_bottom_data=300.0,  # K — temperature at the surface
    interpolation="linear",
)
t_res_aux.to_numpy()[:, :2, :2]

## Writing to GRIB

In [ ]:
t_res.to_target("file", "_pl_to_hl.grib")

# read back and verify
ekd.from_source("file", "_pl_to_hl.grib").to_fieldlist().ls()